# Õppetund 05 - Agentne RAG


## Paigaldus

See märkmik demonstreerib Agentic RAG (otsinguga täiustatud genereerimise) mustrit, kasutades Microsoft Agent raamistiku.

**Eeltingimused:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — teie Azure AI Search teenuse lõpp-punkt
- `AZURE_SEARCH_API_KEY` — teie Azure AI Search API võti
- Azure OpenAI juurutus seadistatud keskkonnamuutujate kaudu
- Azure CLI autentitud (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Mis on Agentne RAG?

Traditsiooniline RAG järgib fikseeritud torujuhtme protsessi: esmalt otsitakse dokumendid, seejärel genereeritakse vastus. **Agentne RAG** läheb kaugemale, andes agendile autonoomia otsustada, **millal** ja **kuidas** teavet otsida.

Agentse RAG-ga saab agent:
- **Otsustada**, kas küsimusele vastamiseks on vaja teha otsing
- **Valida**, millist andmeallikat või tööriista pärida
- **Hinnata** leitud tulemusi ja teha vajadusel täiendavaid otsinguid, kui esimene katse ei ole piisav
- **Kombineerida** teavet mitmest otsinguetapist ühtseks koherentseks vastuseks

See muudab agendi paindlikumaks ja täpsemaks võrreldes staatilise dokumendiotsing-järgse genereerimise torujuhtmega.


## Otsingutööriista loomine

Agentic RAG-is pakitakse välised andmeallikad ümber **tööriistadeks**, mida agent saab vajadusel kutsuda. See võimaldab agendil pidada andmete pärimist lihtsalt üheks tegevuseks, mitte kohustuslikuks sammuks.

Allpool määratleme reisiteadmistebaasi ja avaldame selle tööriistana, mida agent saab kutsuda sihtkoha teabe otsimiseks.


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## RAG-agendi loomine

Nüüd loome agendi, kellele on antud juhis **alati enne vastamist teavet hankida**. Agent kasutab tööriista `search_travel_knowledge`, et oma vastuseid teadmistebaasil toetada, selle asemel et tugineda ainult oma koolitusandmetele.


In [ ]:
agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## Iteratiivne otsing — Maker-Checker mustrit

Agentic RAG üheks oluliseks eeliseks on **iteratiivne otsing**. Agent saab teha mitu otsinguringi, et oma algseid leiukohti kinnitada, täpsustada või laiendada — sarnaselt "maker-checker" töövoo põhimõttele:

1. **Maker samm**: Agent hangib alginfot ja koostab vastuse mustandi.
2. **Checker samm**: Agent teeb täiendavaid otsinguid, et üksikasju kontrollida või lünki täita.

Allpool küsitakse agendilt küsimust, mis nõuab mitme sihtkoha võrdlemist, mis kutsub esile mitu otsingut.


In [ ]:
checker_agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## Kokkuvõte

Selles õppetükis õppisite, kuidas luua **Agentic RAG** süsteemi, kasutades Microsoft Agent Frameworki:

- **Agentic RAG** võimaldab agentidel autonoomselt otsustada, millal teavet otsida, muutes otsimise dünaamiliseks, mitte fikseeritud.
- **Tööriistad kui andmeallikad**: Väliseid teadmistebaase (näiteks Azure AI Search) pakendatakse tööriistadeks, mida agent saab kutsuda.
- **Iteratiivne otsimine**: Maker-checker muster võimaldab agenil teha mitu otsingsammu — otsides, kontrollides ja täiustades — enne lõpliku vastuse andmist.

Tootmiskeskkonnas asendaksite mälus oleva `TRAVEL_KNOWLEDGE_BASE` tegeliku Azure AI Search indeksiga, et hallata suures mahus reisidokumentide otsimist.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Lahtiütlus**:
See dokument on tõlgitud kasutades AI tõlketeenust [Co-op Translator](https://github.com/Azure/co-op-translator). Kuigi me püüdleme täpsuse poole, palun pange tähele, et automatiseeritud tõlgetes võib esineda vigu või ebatäpsusi. Originaaldokument selle emakeeles tuleks pidada autoriteetseks allikaks. Olulise teabe puhul soovitatakse kasutada professionaalset inimtõlget. Me ei vastuta selle tõlkega seotud eksimustest või valesti mõistmistest.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
